In [1]:
import pandas as pd
import numpy as np

# Load your modified dataset
file_path = '../data/merged_dataset_2000_2019.csv'
data = pd.read_csv(file_path)

# Check for initial missing values
missing_values = data.isnull().sum()
print("Initial missing values in the dataset:")
print(missing_values)

Initial missing values in the dataset:
Period                                                                                                                                                        0
ParentLocation                                                                                                                                                0
Location                                                                                                                                                      0
Dim1                                                                                                                                                          0
HALE_Birth                                                                                                                                                    0
HALE_60                                                                                                                                                       0
i

In [2]:
# 1. Filter the data for the period 2000-2019
data_filtered = data[(data['Period'] >= 2000) & (data['Period'] <= 2019)].copy()

# 2. Drop columns with more than 60% missing values
threshold = 0.6 * len(data_filtered)
data_filtered_cleaned = data_filtered.dropna(thresh=threshold, axis=1).copy()

# Show missing values after filtering and dropping columns
missing_after_filter = data_filtered_cleaned.isnull().sum()
print("Missing values after filtering and dropping sparse columns:")
print(missing_after_filter)

Missing values after filtering and dropping sparse columns:
Period                                                                                                                                                        0
ParentLocation                                                                                                                                                0
Location                                                                                                                                                      0
Dim1                                                                                                                                                          0
HALE_Birth                                                                                                                                                    0
HALE_60                                                                                                                                     

In [3]:
# 3. Backward fill for missing values ONLY in the year 2000
for country in data_filtered_cleaned['Location'].unique():
    country_mask = data_filtered_cleaned['Location'] == country
    country_data = data_filtered_cleaned[country_mask]

    mask_2000 = (data_filtered_cleaned['Location'] == country) & (data_filtered_cleaned['Period'] == 2000)

    if not data_filtered_cleaned[mask_2000].empty:
        future_data = country_data[country_data['Period'] > 2000].sort_values('Period')
        if not future_data.empty:
            data_filtered_cleaned.loc[mask_2000] = data_filtered_cleaned.loc[mask_2000].fillna(future_data.iloc[0])

data_filtered_cleaned = data_filtered_cleaned.round(2)

print("Missing values after backward filling year 2000:")
print(data_filtered_cleaned.isnull().sum())

Missing values after backward filling year 2000:
Period                                                                                                                                                        0
ParentLocation                                                                                                                                                0
Location                                                                                                                                                      0
Dim1                                                                                                                                                          0
HALE_Birth                                                                                                                                                    0
HALE_60                                                                                                                                                

In [4]:
import pandas as pd
import numpy as np

# 1. Separate numeric columns and categorical columns
# This prevents the 'object dtype' warning
numeric_cols = data_filtered_cleaned.select_dtypes(include=[np.number]).columns
categorical_cols = data_filtered_cleaned.select_dtypes(exclude=[np.number]).columns

# 2. Apply interpolation ONLY on numeric columns, grouped by Location
# This ensures each country's gaps are filled based on its own trend
data_interpolated_numeric = data_filtered_cleaned.groupby('Location', group_keys=False)[numeric_cols].apply(
    lambda group: group.interpolate(method='linear')
)

# 3. Combine the numeric results back with the categorical columns
data_interpolated = pd.concat([data_filtered_cleaned[categorical_cols], data_interpolated_numeric], axis=1)

# 4. Final step: Round values and check for any remaining nulls
data_interpolated = data_interpolated.round(2)

print("Success! Missing values after fixed interpolation:")
print(data_interpolated.isnull().sum())

Success! Missing values after fixed interpolation:
ParentLocation                                                                                                                                                0
Location                                                                                                                                                      0
Dim1                                                                                                                                                          0
Period                                                                                                                                                        0
HALE_Birth                                                                                                                                                    0
HALE_60                                                                                                                                              

In [5]:
# 5. Global Mean Imputation for any remaining missing values
data_final = data_interpolated.fillna(data_interpolated.mean(numeric_only=True))
data_final = data_final.round(2)

print("Final check for missing values (Should be 0):")
print(data_final.isnull().sum())

Final check for missing values (Should be 0):
ParentLocation                                                                                                                                             0
Location                                                                                                                                                   0
Dim1                                                                                                                                                       0
Period                                                                                                                                                     0
HALE_Birth                                                                                                                                                 0
HALE_60                                                                                                                                                    0
infant morta

In [6]:
# 6. Save the final cleaned dataset
output_file = '../data/cleaned_merged_dataset.csv'
data_final.to_csv(output_file, index=False)

print(f"Data preprocessing complete! The cleaned dataset is saved as '{output_file}'.")

Data preprocessing complete! The cleaned dataset is saved as '../data/cleaned_merged_dataset.csv'.
